# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and initialize
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

In [ ]:
# List all record sets, fields, and columns with their @ids
print('--- Record Sets ---')
for rs in metadata.record_sets:
    print('RecordSet @id:', rs.id, '| name:', rs.name)
    print('  Fields:')
    for f in rs.fields:
        print('    Field @id:', f.id, '| name:', f.name, '| type:', getattr(f, 'data_type', None))
        # If columns are defined
        if hasattr(f, 'columns') and f.columns:
            for c in f.columns:
                print('      Column @id:', c.id, '| name:', getattr(c, 'name', None))
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# --- Identify record sets for extraction (by @id) ---
record_sets = [rs.id for rs in metadata.record_sets]
print('Record Sets:')
for x in record_sets:
    print(' ', x)

dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

# Show columns in the first available record set
if record_sets:
    print(f"Columns in {record_sets[0]}:", dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

In [ ]:
# Choose a numeric field for analysis by inspecting the DataFrame columns above.
# For demonstration, let's select the first numeric column automatically.

import numpy as np

record_set_id = record_sets[0] if record_sets else None
df = dataframes[record_set_id]

numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    print("No numeric field found for EDA.")
else:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records in '{record_set_id}' with '{numeric_field}' > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    normalized = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = normalized
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick another (non-numeric) field for grouping
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
        print(f"\nGrouped data in '{record_set_id}' by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship with the group field (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Histogram for numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"'{numeric_field}' grouped by '{group_field}' in '{record_set_id}'")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
This notebook demonstrated how to load, inspect, and process the FAIR^2 mass spectrometry dataset using `mlcroissant`.

- We inspected the Croissant schema and reviewed fields and columns by their `@id`.
- Extracted tabular data for analysis and demonstrated simple filtering, normalization, and grouping.
- Generated basic visualizations for data understanding.

**Next steps:** Deeper biological or domain-specific analysis can be performed depending on research questions or hypotheses.